In [ ]:
from pyGmsh import pyGmsh
import numpy as np
import openseespy.opensees as ops

# Shell I-Beam Modal Analysis — IPE200 Cantilever

**Goal:** Mesh an IPE200 steel profile (3 m long) with shell elements, compute
natural frequencies and mode shapes using OpenSees eigenvalue analysis, and
compare with Euler–Bernoulli beam theory.

**Architecture:** Same decoupled pattern as the plate example:

1. **pyGmsh** builds the geometry, generates a structured quad mesh, and
   provides post-processing views.
2. **OpenSeesPy** receives plain numpy arrays, builds the shell model, and
   runs the eigenvalue solver.
3. **Results flow back** to Gmsh views for interactive 3D exploration.

**Shell modelling strategy:**

An I-beam has three flat plates (two flanges + one web). We model their
**mid-surfaces** as 2D shell elements living in 3D space. The flanges and
web meet at shared edges, which gives us a conforming mesh at the junctions.
Each surface gets its own section thickness (t_f for flanges, t_w for web).

In [ ]:
# ============================================================
# IPE200 cross-section (European standard)
# ============================================================
#
#          ←── b ──→
#    ┌─────────────────┐  ─┬─ tf
#    └───────┐ ┌───────┘   │
#            │ │  tw        │
#            │ │            │ h = 200 mm
#            │ │            │
#    ┌───────┘ └───────┐   │
#    └─────────────────┘  ─┴─ tf
#
# For the shell model we use mid-surface representation:
#   - Flange mid-surfaces at z = 0 and z = d
#   - Web mid-surface connects them at y = 0
#   - d = h - tf (distance between flange centerlines)

h   = 200.0     # [mm] total depth
b   = 100.0     # [mm] flange width
tw  = 5.6       # [mm] web thickness
tf  = 8.5       # [mm] flange thickness
L   = 3000.0    # [mm] beam length (cantilever)

d = h - tf      # [mm] distance between flange mid-surfaces = 191.5

# Material properties — steel (consistent units: mm, N, s, tonne)
E   = 210000.0    # [MPa = N/mm²]
nu  = 0.3         # [-]
rho = 7.85e-9     # [tonne/mm³]  (7850 kg/m³ in mm-N-s-tonne system)

# Cross-section properties for analytical comparison
A_cs  = 2 * b * tf + (h - 2*tf) * tw              # [mm²] cross-section area
Iy_cs = (b*h**3 - (b-tw)*(h-2*tf)**3) / 12        # [mm⁴] strong-axis I
Iz_cs = (2*tf*b**3 + (h-2*tf)*tw**3) / 12         # [mm⁴] weak-axis I

print(f"IPE200 shell model: L = {L:.0f} mm, d = {d:.1f} mm")
print(f"A  = {A_cs:.1f} mm²")
print(f"Iy = {Iy_cs:.0f} mm⁴  (strong axis)")
print(f"Iz = {Iz_cs:.0f} mm⁴  (weak axis)")

## Part 1 — Geometry: I-beam mid-surfaces

We build five rectangular surfaces sharing edges at the web–flange junctions:

```
    Coordinate system:
    X = beam axis [0, L]
    Y = flange width [-b/2, b/2]
    Z = depth [0, d]

    Cross-section at x = 0:

      p5────────p4────────p6     z = d  (top flange)
                |
                |  web
                |
      p1────────p2────────p3     z = 0  (bottom flange)
     y=-b/2    y=0       y=b/2
```

Each flange is split into two halves (left and right of the web) so the
mesh nodes along the junction line are shared. This creates 5 surfaces:
BF_left, BF_right, Web, TF_left, TF_right.

In [ ]:
g = pyGmsh(model_name="IPE200_Shell", verbose=True)
g.initialize()

# --- 12 corner points of the I-beam mid-surface ---
#
# Near end (x = 0):
p1  = g.model.add_point(0, -b/2, 0,    lc=30)   # bot-flange left
p2  = g.model.add_point(0,  0,   0,    lc=30)   # bot-flange center (junction)
p3  = g.model.add_point(0,  b/2, 0,    lc=30)   # bot-flange right
p4  = g.model.add_point(0,  0,   d,    lc=30)   # top-flange center (junction)
p5  = g.model.add_point(0, -b/2, d,    lc=30)   # top-flange left
p6  = g.model.add_point(0,  b/2, d,    lc=30)   # top-flange right

# Far end (x = L):
p7  = g.model.add_point(L, -b/2, 0,    lc=30)
p8  = g.model.add_point(L,  0,   0,    lc=30)
p9  = g.model.add_point(L,  b/2, 0,    lc=30)
p10 = g.model.add_point(L,  0,   d,    lc=30)
p11 = g.model.add_point(L, -b/2, d,    lc=30)
p12 = g.model.add_point(L,  b/2, d,    lc=30)

# --- Cross-section lines at x = 0 ---
c1 = g.model.add_line(p1, p2, label="bf_l_near")    # bot-flange left half
c2 = g.model.add_line(p2, p3, label="bf_r_near")    # bot-flange right half
c3 = g.model.add_line(p2, p4, label="web_near")     # web
c4 = g.model.add_line(p5, p4, label="tf_l_near")    # top-flange left half (p5→p4)
c5 = g.model.add_line(p4, p6, label="tf_r_near")    # top-flange right half

# --- Cross-section lines at x = L ---
c6  = g.model.add_line(p7,  p8,  label="bf_l_far")
c7  = g.model.add_line(p8,  p9,  label="bf_r_far")
c8  = g.model.add_line(p8,  p10, label="web_far")
c9  = g.model.add_line(p11, p10, label="tf_l_far")
c10 = g.model.add_line(p10, p12, label="tf_r_far")

# --- Longitudinal lines (beam axis) ---
e1 = g.model.add_line(p1,  p7,  label="edge_bl")    # bottom-left edge
e2 = g.model.add_line(p2,  p8,  label="junction_b") # bottom junction
e3 = g.model.add_line(p3,  p9,  label="edge_br")    # bottom-right edge
e4 = g.model.add_line(p4,  p10, label="junction_t") # top junction
e5 = g.model.add_line(p5,  p11, label="edge_tl")    # top-left edge
e6 = g.model.add_line(p6,  p12, label="edge_tr")    # top-right edge

# --- 5 Surfaces ---
#
# Each surface is a planar quad defined by 4 edges in a closed loop.
# Negative sign = reversed curve direction.

# Bottom flange — left half (z = 0, y ∈ [-b/2, 0])
loop_bf_l = g.model.add_curve_loop([c1, e2, -c6, -e1])
s_bf_l    = g.model.add_plane_surface(loop_bf_l, label="BotFlange_L")

# Bottom flange — right half (z = 0, y ∈ [0, b/2])
loop_bf_r = g.model.add_curve_loop([c2, e3, -c7, -e2])
s_bf_r    = g.model.add_plane_surface(loop_bf_r, label="BotFlange_R")

# Web (y = 0, z ∈ [0, d])
loop_web  = g.model.add_curve_loop([c3, e4, -c8, -e2])
s_web     = g.model.add_plane_surface(loop_web, label="Web")

# Top flange — left half (z = d, y ∈ [-b/2, 0])
loop_tf_l = g.model.add_curve_loop([-c4, e5, c9, -e4])
s_tf_l    = g.model.add_plane_surface(loop_tf_l, label="TopFlange_L")

# Top flange — right half (z = d, y ∈ [0, b/2])
loop_tf_r = g.model.add_curve_loop([c5, e6, -c10, -e4])
s_tf_r    = g.model.add_plane_surface(loop_tf_r, label="TopFlange_R")

g.model.registry()

In [ ]:
# --- Physical groups ---
#
# Surfaces → element regions (different shell thickness)
# Curves   → boundary nodes (fixed end at x = 0)

pg_flanges = g.physical.add(2, [s_bf_l, s_bf_r, s_tf_l, s_tf_r], name="Flanges")
pg_web     = g.physical.add(2, [s_web], name="Web")

# Fixed end: all cross-section curves at x = 0
pg_fixed = g.physical.add(1, [c1, c2, c3, c4, c5], name="FixedEnd")

g.physical.summary()

In [ ]:
import gmsh

# --- Transfinite meshing for structured quad mesh ---
#
# All 5 surfaces are rectangles → perfect for transfinite.
# We set the number of NODES (= divisions + 1) on each curve,
# classified by geometric length.

n_half_flange = 4    # divisions across half-flange width (50 mm)
n_web_height  = 12   # divisions across web height (d ≈ 191.5 mm)
n_length      = 60   # divisions along beam length (3000 mm)

# Cross-section curves: flange half-width
for c in [c1, c2, c4, c5, c6, c7, c9, c10]:
    gmsh.model.mesh.setTransfiniteCurve(c, n_half_flange + 1)

# Cross-section curves: web height
for c in [c3, c8]:
    gmsh.model.mesh.setTransfiniteCurve(c, n_web_height + 1)

# Longitudinal curves: beam length
for c in [e1, e2, e3, e4, e5, e6]:
    gmsh.model.mesh.setTransfiniteCurve(c, n_length + 1)

# Mark all surfaces as transfinite + recombine (→ quads)
for s in [s_bf_l, s_bf_r, s_web, s_tf_l, s_tf_r]:
    gmsh.model.mesh.setTransfiniteSurface(s)
    gmsh.model.mesh.setRecombine(2, s)

# Generate 2D mesh (shell surfaces in 3D space)
g.mesh.set_order(1)
g.mesh.generate(2)

# Quick stats
stats = g.mesh.get_elements(dim=2)
n_quads = sum(len(t) for t in stats['tags'])
print(f"Generated {n_quads} quad elements")

## Part 2 — Extract mesh data

We need:
1. **All nodes and elements** via `get_fem_data(dim=2)`
2. **Region classification** — which elements are flanges vs web (for different section thickness)
3. **Fixed-end nodes** — from the physical group at x = 0

In [ ]:
# --- All mesh arrays in one call ---
fem = g.mesh.get_fem_data(dim=2)

node_tags    = fem.node_ids
node_coords  = fem.node_coords
tag_to_idx   = {int(t): i for i, t in enumerate(fem.node_ids)}
connectivity = fem.connectivity    # (nElem, 4) for quads
elem_tags    = fem.element_ids
used_tags    = set(fem.node_ids.tolist())

# --- Region classification: flange vs web elements ---
#
# We query elements per surface entity to build lookup sets.
# This tells us which section thickness to assign each element.

flange_elem_set = set()
for surf in [s_bf_l, s_bf_r, s_tf_l, s_tf_r]:
    elems = g.mesh.get_elements(dim=2, tag=surf)
    for etags in elems['tags']:
        flange_elem_set.update(etags.astype(int).tolist())

web_elem_set = set()
elems_web = g.mesh.get_elements(dim=2, tag=s_web)
for etags in elems_web['tags']:
    web_elem_set.update(etags.astype(int).tolist())

# --- Fixed-end nodes (all DOFs constrained) ---
fixed_nodes = g.physical.get_nodes(1, pg_fixed)['tags']

print(f"Nodes: {len(node_tags)} total, {len(used_tags)} used")
print(f"Quads: {connectivity.shape[0]}  ({len(flange_elem_set)} flange + {len(web_elem_set)} web)")
print(f"Fixed-end nodes: {len(fixed_nodes)}")

## Part 3 — OpenSees Shell Model

**Shell element choice: `ShellMITC4`**

MITC4 (Mixed Interpolation of Tensorial Components) is a 4-node shell
element that avoids shear locking by using a mixed interpolation for
the transverse shear strains. It supports both membrane and bending
behaviour — exactly what we need for an I-beam where:
- Flanges carry mostly bending/membrane stresses
- Web carries mostly shear

**Section: `ElasticMembranePlateSection`**

Combines membrane (in-plane) and plate (bending) behaviour for an
elastic isotropic material. Parameters: E, ν, thickness, density.
We use two sections — one with t_f for flanges, one with t_w for the web.

**DOFs:** 6 per node (u_x, u_y, u_z, θ_x, θ_y, θ_z) — standard for shells in 3D.

In [ ]:
ops.wipe()
ops.model('basic', '-ndm', 3, '-ndf', 6)

# --- Nodes ---
# Map Gmsh tags → sequential OpenSees IDs, skipping orphan nodes.

gmsh_to_ops = {}
new_id = 0
for gtag, coords in zip(node_tags.astype(int), node_coords):
    if int(gtag) not in used_tags:
        continue
    new_id += 1
    gmsh_to_ops[int(gtag)] = new_id
    ops.node(new_id, float(coords[0]), float(coords[1]), float(coords[2]))

print(f"Created {new_id} OpenSees nodes")

# --- Sections ---
#
# ElasticMembranePlateSection(tag, E, nu, h, rho)
#   - h = shell thickness
#   - rho = mass density per unit volume (needed for eigenvalue analysis)

sec_flange = 1
sec_web    = 2
ops.section('ElasticMembranePlateSection', sec_flange, E, nu, tf, rho)
ops.section('ElasticMembranePlateSection', sec_web,    E, nu, tw, rho)

# --- Elements ---
#
# ShellMITC4(tag, n1, n2, n3, n4, secTag)
# Node ordering must follow the element's local coordinate convention
# (counter-clockwise when viewed from the positive normal direction).

n_flange_elems = 0
n_web_elems = 0

for i, (etag, row) in enumerate(zip(elem_tags, connectivity)):
    eid = i + 1
    nodes = [gmsh_to_ops[int(n)] for n in row]
    
    if etag in flange_elem_set:
        ops.element('ShellMITC4', eid, *nodes, sec_flange)
        n_flange_elems += 1
    elif etag in web_elem_set:
        ops.element('ShellMITC4', eid, *nodes, sec_web)
        n_web_elems += 1
    else:
        # Should not happen — every element belongs to a surface
        print(f"WARNING: element {etag} not classified!")

print(f"Created {n_flange_elems} flange + {n_web_elems} web = {n_flange_elems + n_web_elems} ShellMITC4 elements")

# --- Boundary conditions ---
#
# Cantilever: fix ALL 6 DOFs at x = 0

n_fixed = 0
for gtag in fixed_nodes.astype(int):
    if int(gtag) in gmsh_to_ops:
        ops.fix(gmsh_to_ops[int(gtag)], 1, 1, 1, 1, 1, 1)
        n_fixed += 1

print(f"Fixed {n_fixed} nodes at x = 0 (cantilever)")

## Part 4 — Eigenvalue Analysis

OpenSees eigenvalue solver finds the first N natural frequencies and
mode shapes. We use `ops.eigen(N)` which returns the eigenvalues
λ_n = ω_n² (angular frequency squared).

Natural frequency: f_n = ω_n / (2π) = √λ_n / (2π)  [Hz]

For a cantilever I-beam, we expect to see:
1. **1st bending — weak axis** (lateral sway, lowest frequency)
2. **1st bending — strong axis** (vertical deflection)
3. **1st torsional mode**
4. Higher harmonics of the above

In [ ]:
# --- Eigenvalue analysis setup ---
ops.constraints('Transformation')
ops.numberer('RCM')
ops.system('FullGeneral')     # FullGeneral for eigenvalue problems
ops.algorithm('Linear')

# Compute first 10 modes
num_modes = 10
eigenvalues = ops.eigen(num_modes)

# Convert to natural frequencies
frequencies = []
periods = []
print(f"{'Mode':>4s}  {'f [Hz]':>10s}  {'T [s]':>10s}  {'ω [rad/s]':>12s}")
print("-" * 42)
for i, ev in enumerate(eigenvalues):
    omega = np.sqrt(ev)
    f = omega / (2 * np.pi)
    T = 1.0 / f if f > 0 else float('inf')
    frequencies.append(f)
    periods.append(T)
    print(f"{i+1:4d}  {f:10.3f}  {T:10.4f}  {omega:12.3f}")

In [ ]:
# --- Extract mode shapes ---
#
# For each mode, we read the eigenvector at every node.
# ops.nodeEigenvector(nodeTag, mode, dof) returns the modal displacement.
#
# We store mode shapes as (nNode, 6) arrays aligned with node_coords
# (using tag_to_idx for indexing, same as the plate example).

nNode = len(node_tags)
mode_shapes = []    # list of (nNode, 6) arrays — one per mode

for mode in range(1, num_modes + 1):
    phi = np.zeros((nNode, 6))
    for gtag, ops_id in gmsh_to_ops.items():
        idx = tag_to_idx[gtag]
        for dof in range(6):
            phi[idx, dof] = ops.nodeEigenvector(ops_id, mode, dof + 1)
    mode_shapes.append(phi)

# --- Done with OpenSees ---
ops.wipe()
print(f"Extracted {num_modes} mode shapes ({nNode} nodes × 6 DOFs each)")

## Part 5 — Analytical Comparison (Euler–Bernoulli)

For a cantilever beam of length L, the natural frequencies from
Euler–Bernoulli theory are:

$$f_n = \frac{(\beta_n L)^2}{2\pi L^2} \sqrt{\frac{EI}{\rho A}}$$

where β_n L are the roots of cos(βL)·cosh(βL) + 1 = 0:
β₁L = 1.8751,  β₂L = 4.6941,  β₃L = 7.8548

We compute these for both the strong axis (I_y) and weak axis (I_z) to
identify which FEM modes correspond to which analytical prediction.

In [ ]:
# --- Euler-Bernoulli cantilever beam frequencies ---

beta_L = [1.87510, 4.69409, 7.85476, 10.9955, 14.1372]  # first 5 roots

def euler_bernoulli_freq(beta_n_L, E, I, rho, A, L):
    """Natural frequency [Hz] for cantilever beam mode."""
    omega_sq = (beta_n_L / L)**4 * (E * I) / (rho * A)
    return np.sqrt(omega_sq) / (2 * np.pi)

print("Euler-Bernoulli analytical frequencies:")
print(f"{'n':>3s}  {'f_strong [Hz]':>14s}  {'f_weak [Hz]':>14s}")
print("-" * 36)

f_strong_analytical = []
f_weak_analytical = []
for i, bL in enumerate(beta_L):
    f_s = euler_bernoulli_freq(bL, E, Iy_cs, rho, A_cs, L)
    f_w = euler_bernoulli_freq(bL, E, Iz_cs, rho, A_cs, L)
    f_strong_analytical.append(f_s)
    f_weak_analytical.append(f_w)
    print(f"{i+1:3d}  {f_s:14.3f}  {f_w:14.3f}")

print()
print("Note: Shell model also captures torsional modes and")
print("local flange/web modes not predicted by beam theory.")

## Part 6 — Mode Shape Visualization

We plot the first 6 mode shapes as 3D wireframes using matplotlib,
showing the deformed shape scaled for visibility.

In [ ]:
import matplotlib.pyplot as plt
from mpl_toolkits.mplot3d.art3d import Poly3DCollection

fig, axes = plt.subplots(2, 3, figsize=(18, 10),
                         subplot_kw={'projection': '3d'})

# Connectivity in terms of array indices (for plotting)
conn_idx = np.array([[tag_to_idx[int(n)] for n in row] for row in connectivity])

for mode_idx, ax in enumerate(axes.flat):
    if mode_idx >= num_modes:
        break
    
    phi = mode_shapes[mode_idx]
    
    # Auto-scale: normalize so max displacement = 10% of beam length
    max_disp = np.max(np.abs(phi[:, :3]))
    scale = 0.10 * L / max_disp if max_disp > 1e-15 else 1.0
    
    # Deformed coordinates
    deformed = node_coords.copy()
    deformed[:, 0] += scale * phi[:, 0]
    deformed[:, 1] += scale * phi[:, 1]
    deformed[:, 2] += scale * phi[:, 2]
    
    # Compute displacement magnitude for coloring
    disp_mag = np.sqrt(phi[:, 0]**2 + phi[:, 1]**2 + phi[:, 2]**2)
    
    # Plot deformed quad mesh as wireframe
    for row in conn_idx:
        verts = deformed[row]
        # Close the quad
        quad = np.vstack([verts, verts[0:1]])
        ax.plot(quad[:, 0], quad[:, 1], quad[:, 2],
                color='steelblue', lw=0.3, alpha=0.6)
    
    ax.set_title(f"Mode {mode_idx+1}: f = {frequencies[mode_idx]:.2f} Hz",
                 fontsize=10)
    ax.set_xlabel("X [mm]")
    ax.set_ylabel("Y [mm]")
    ax.set_zlabel("Z [mm]")
    ax.view_init(elev=25, azim=-60)

plt.tight_layout()
plt.show()

## Part 7 — Gmsh Post-processing Views

Pass mode shapes back to pyGmsh for interactive 3D visualization.
Each mode becomes a vector view that you can animate in the Gmsh GUI.

In [ ]:
# --- Create Gmsh views for each mode shape ---
#
# We pass the translational DOFs (ux, uy, uz) as node vectors.
# In the Gmsh GUI you can then:
#   - View > Displacement Factor to scale the deformation
#   - Tools > Options > View > Vector Display to choose arrow vs deformation

node_tag_list = list(node_tags.astype(int))

for mode_idx in range(min(num_modes, 6)):
    phi = mode_shapes[mode_idx]
    # Translational DOFs only (columns 0,1,2)
    g.view.add_node_vector(
        f"Mode {mode_idx+1} — f={frequencies[mode_idx]:.2f} Hz",
        node_tag_list,
        phi[:, :3]
    )

# Displacement magnitude for mode 1 (scalar for smooth contour)
phi1 = mode_shapes[0]
mag1 = np.sqrt(phi1[:, 0]**2 + phi1[:, 1]**2 + phi1[:, 2]**2)
g.view.add_node_scalar("Mode 1 |u|", node_tag_list, mag1)

print(f"Created {g.view.count()} Gmsh views")

In [ ]:
# --- Summary: FEM vs Analytical ---
#
# Match FEM modes to analytical predictions by inspecting dominant
# displacement direction. This is a rough classification:
#   - Dominant uy → weak-axis bending
#   - Dominant uz → strong-axis bending
#   - Dominant θx (rotation about beam axis) → torsional

print("\nMode Classification (first 10 FEM modes):")
print(f"{'Mode':>4s}  {'f_FEM [Hz]':>10s}  {'Dominant DOF':>14s}  {'Type':>20s}")
print("-" * 55)

for i in range(num_modes):
    phi = mode_shapes[i]
    # RMS of each translational DOF across all nodes
    rms = [np.sqrt(np.mean(phi[:, j]**2)) for j in range(6)]
    
    # Classify by dominant DOF
    max_dof = np.argmax(rms[:3])  # 0=ux, 1=uy, 2=uz
    
    if max_dof == 1:
        mode_type = "Weak-axis bending"
    elif max_dof == 2:
        mode_type = "Strong-axis bending"
    elif max_dof == 0:
        mode_type = "Axial / coupled"
    else:
        mode_type = "Unknown"
    
    # Check if torsion dominates (rotation about x)
    if rms[3] > 1.5 * max(rms[:3]):
        mode_type = "Torsional"
    
    dof_labels = ['ux', 'uy', 'uz', 'θx', 'θy', 'θz']
    dom_label = dof_labels[np.argmax(rms)]
    
    print(f"{i+1:4d}  {frequencies[i]:10.3f}  {dom_label:>14s}  {mode_type:>20s}")

## Part 8 — Launch Gmsh GUI & Finalize

In [ ]:
# Launch Gmsh GUI — explore mode shapes interactively
# Use View > Displacement Factor to exaggerate the deformation
g.launch_gui()

In [ ]:
g.finalize()